In [70]:
import os, sys
import requests
week2_path = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0,week2_path)
# import scraper.py
from scraper import fetch_website_contents
# import environment variables module
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

In [71]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


OpenAI API Key exists and begins sk-proj-
Anthropic API Key not set (and this is optional)
Google API Key exists and begins AI
DeepSeek API Key exists and begins sk-
Groq API Key exists and begins gsk_
Grok API Key not set (and this is optional)
OpenRouter API Key exists and begins sk-


In [93]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

# For Gemini, DeepSeek and Groq, we can use the OpenAI python client
# Because Google and DeepSeek have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [72]:
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

In [73]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

Why did the aspiring LLM engineer bring a ladder to the data center?

Because they wanted to reach the next level... of model training!

## Training vs Inference time scaling

In [74]:
easy_puzzle = [
    {"role": "user", "content": 
        "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."},
]

In [75]:
response = openai.chat.completions.create(model="gpt-5-nano", messages=easy_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))

1/3

In [76]:
response = openai.chat.completions.create(model="gpt-5-nano", messages=easy_puzzle, reasoning_effort="low")
display(Markdown(response.choices[0].message.content))

2/3

In [77]:
response = openai.chat.completions.create(model="gpt-5-mini", messages=easy_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))

2/3

## Testing out the best models on the planet

In [78]:
hard = """
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.
What distance did it gnaw through?
"""
hard_puzzle = [
    {"role": "user", "content": hard}
]

In [11]:
response = openai.chat.completions.create(model="gpt-5-nano", messages=hard_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))

We have two volumes side by side on a shelf. Each volume has:
- total pages thickness: 2 cm
- each cover thickness: 2 mm = 0.2 cm

A worm gnaws perpendicular to the pages from the first page of the first volume to the last page of the second volume.

We need the distance through which the worm ate, i.e., the material between those two pages along a straight line through the books.

Key facts about book orientation:
- When books are shelved upright, the pages run vertical, and opening is along the vertical axis. The thickness of a volume is determined by the stack of pages plus covers.
- The first page of the first volume is at the left edge of that volume’s pages (i.e., near the front cover if we view the book from the side). The last page of the second volume is at the right edge of that volume’s pages (near the back cover).

However, the worm’s path is perpendicular to the pages, i.e., straight through the thickness dimension.

We consider the overall arrangement from left to right:
- Volume 1: front cover thickness 0.2 cm, pages 2 cm total, back cover 0.2 cm. Total thickness = 0.2 + 2 + 0.2 = 2.4 cm.
- Then Volume 2: front cover 0.2 cm, pages 2 cm, back cover 0.2 cm. Total 2.4 cm.

When the volumes are on the shelf side by side, the order is:
[Front cover of Vol 1] ... [Back cover Vol 1] [Front cover Vol 2] ... [Back cover Vol 2]

The worm starts at the first page of the first volume. The first page is immediately after the front cover of Volume 1. So its position is just inside the Vol 1 near the left side of the page block.

The worm ends at the last page of the second volume, which is just before the back cover of Volume 2 (the far right side of the pages of Vol 2).

So the worm’s path runs through:
- The remainder of Volume 1 after the first page: from page 1 boundary to the back of Vol 1’s pages, i.e., through the rest of the Volume 1 (including its back cover? careful: the path is through material, but ends at the last page of Vol 2, not through Vol 2’s back cover). Since it ends at the last page of Vol 2, the path does not go through Vol 2’s back cover, but ends at the back side of the last page.

Thus the worm travels through:
- The remainder of Volume 1 after the first page: include the rest of Vol 1 pages and Vol 1 back cover? The end is at last page of Vol 2, which is after crossing the entire Volume 1 and the full Volume 2 front-to-page region up to the last page.

More simply: The distance from the first page of Vol 1 to the last page of Vol 2, measured perpendicular to pages, equals:
- the thickness of the portion between those two pages: that includes the rest of Vol 1 after the first page, plus the front cover of Vol 2, plus all pages of Vol 2 up to the last page.

Compute components:
- In Vol 1, from the first page to the back cover: pages remaining + back cover thickness = (total Vol 1 thickness) - (front cover) - (first page thickness before first page?). But "first page" is at the very start of the page block, immediately after the front cover. So after the first page, there is the rest of the pages (2 cm total) minus the portion that is before the first page. The first page has some thickness (negligible). In standard puzzles, the first page starts right after the front cover, so the distance from the first page to the end of Vol 1 equals the entire page block plus back cover: 2 cm (pages) + 0.2 cm (back cover) = 2.2 cm.

- Then the front cover of Vol 2: 0.2 cm.

- Then the pages of Vol 2 up to the last page: that's all the pages in Vol 2 except the back cover. The last page is at the end of the page block, just before the back cover. So from the first page of Vol 2 to the last page is basically the entire page block, i.e., 2 cm.

But careful: the worm starts at the first page of Vol 1 (the interior surface of that page). The distance to the last page of Vol 2 includes traversing through front cover of Vol 1? No, because the starting point is after the front cover of Vol 1, at the first page. So we do not traverse front cover of Vol 1.

We travel through:
- The rest of Vol 1: pages (2 cm total pages) minus the thickness of the first page itself. If we treat the first page as part of the 2 cm pages, the distance from first page to the end of Vol 1's page block is approximately 2 cm (the entire page block) minus the negligible thickness of the first page before the starting point. For puzzles like this, we take it as the rest of the Vol 1 pages plus back cover: 2 cm (pages remaining) + 0.2 cm (back cover) = 2.2 cm.

But do we include the back cover of Vol 1? Yes, to reach the front of Vol 2, you pass through Vol 1 back cover.

- Then front cover Vol 2: 0.2 cm.

- Then through Vol 2 pages up to last page: 2 cm.

Sum: 2.2 + 0.2 + 2 = 4.4 cm.

Answer: 4.4 cm.

In [79]:
response = openai.chat.completions.create(model="gpt-5", messages=hard_puzzle)
display(Markdown(response.choices[0].message.content))

4 mm.

Reason: On a shelf (spines outward), the side of Volume 1 facing Volume 2 is its front cover; the side of Volume 2 facing Volume 1 is its back cover. The first page of Vol. 1 lies just inside its front cover, and the last page of Vol. 2 lies just inside its back cover. So the worm passes only through these two facing covers: 2 mm + 2 mm = 4 mm, and through no pages.

## A spicy challenge to test the competitive spirit

In [80]:
dilemma_prompt = """
You and a partner are contestants on a game show. You're each taken to separate rooms and given a choice:
Cooperate: Choose "Share" — if both of you choose this, you each win $1,000.
Defect: Choose "Steal" — if one steals and the other shares, the stealer gets $2,000 and the sharer gets nothing.
If both steal, you both get nothing.
Do you choose to Steal or Share? Pick one.
"""

dilemma = [
    {"role": "user", "content": dilemma_prompt},
]


In [81]:
response = groq.chat.completions.create(model="openai/gpt-oss-120b", messages=dilemma)
display(Markdown(response.choices[0].message.content))

I’d pick **Share**.

**Why?**  

- If both players cooperate (both “Share”), each walks away with $1,000 – a solid, guaranteed payoff.  
- If I “Steal” while you “Share,” I’d get $2,000, but that depends on you trusting me enough to share. In the high‑stakes, one‑shot setting, the safest way to ensure a positive outcome for both is to cooperate.  
- If both of us “Steal,” we both end up with nothing, which is the worst possible result for each of us. By choosing “Share,” I at least avoid the risk of mutual loss, and I give the other player a clear incentive to reciprocate.

So, my choice is **Share**.

In [82]:
response = deepseek.chat.completions.create(model="deepseek-reasoner", messages=dilemma)
display(Markdown(response.choices[0].message.content))

Steal.

In [85]:
response = gemini.chat.completions.create(model="gemini-2.5-flash", messages=dilemma)
display(Markdown(response.choices[0].message.content))

This is a classic Prisoner's Dilemma scenario. Let's analyze the outcomes from my perspective:

1.  **If my partner chooses "Share":**
    *   If I choose "Share": I get $1,000.
    *   If I choose "Steal": I get $2,000. (Better for me)

2.  **If my partner chooses "Steal":**
    *   If I choose "Share": I get $0.
    *   If I choose "Steal": I get $0. (No worse for me)

From a purely rational, self-interested perspective, "Steal" is the dominant strategy. No matter what my partner chooses, I either do better or no worse by choosing "Steal."

However, if both of us choose "Steal" (the rational choice individually), we both end up with $0, which is the worst collective outcome. If both of us chose "Share" (the cooperative choice), we would both win $1,000. The dilemma is that trusting the other person to cooperate leaves me vulnerable to getting nothing while they take $2,000.

Given that I don't know my partner and this is a one-shot game, the logical, self-preserving choice to avoid the worst individual outcome (getting nothing while my partner gets $2,000) is to "Steal."

Therefore, I choose to **Steal**.

## Going local

Just use the OpenAI library pointed to localhost:11434/v1

In [ ]:
requests.get("http://localhost:11434/").content

b'Ollama is running'

In [ ]:
!"C:\Users\jagad\AppData\Local\Programs\Ollama\ollama.exe" pull llama3.2

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [ ]:
!"C:\Users\jagad\AppData\Local\Programs\Ollama\ollama.exe" pull gpt-oss:20b

^C


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏  62 KB/ 13 GB                  pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏ 752 KB/ 13 GB                  pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏ 6.1 MB/ 13 GB                  pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏  11 MB/ 13 GB                  pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏  13 MB/ 13 GB                  pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏  18 MB/ 13 GB                  pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏  24 MB/ 13 GB                  pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏  28 MB/ 13 GB                  pulling manifest 
pulling e7b273f96360:   0% ▕       

In [ ]:
response = ollama.chat.completions.create(model="llama3.2", messages=easy_puzzle)
display(Markdown(response.choices[0].message.content))

1/2

# Gemini and groq python client libraries

In [13]:
from google import genai

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash-lite", contents="Describe the color Blue to someone who's never been able to see in 1 sentence"
)
print(response.text)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Blue is the cool, calming feeling of a clear sky on a sunny day, or the deep, vast mystery of the ocean.


## Routers and Abtraction Layers

In [ ]:
response = openrouter.chat.completions.create(model="z-ai/glm-4.5", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

APIStatusError: Error code: 402 - {'error': {'message': 'Insufficient credits. This account never purchased credits. Make sure your key is on the correct account or org, and if so, purchase more at https://openrouter.ai/settings/credits', 'code': 402}}

: 

# Langchain

In [14]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-mini")
response = llm.invoke(tell_a_joke)

display(Markdown(response.content))

How many LLM engineering students does it take to change a lightbulb?

None — they’ll just prompt the model to hallucinate sunlight.

#LiteLLM

In [15]:
from litellm import completion
response = completion(model="openai/gpt-4.1", messages=tell_a_joke)
reply = response.choices[0].message.content
display(Markdown(reply))

Why did the LLM engineering student get kicked out of the library?

Because every time they tried to summarize a book, they hallucinated the ending!

In [16]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 24
Output tokens: 30
Total tokens: 54
Total cost: 0.0288 cents


In [35]:
with open("hamlet.txt", "r", encoding="utf-8") as f:
    hamlet = f.read()

loc = hamlet.find("Speak, man")
print(hamlet[loc:loc+100])

Speak, man.
  Laer. Where is my father?
  King. Dead.
  Queen. But not by him!
  King. Let him deman


In [36]:
question = [{"role": "user", "content": "In Hamlet, when Laertes asks 'Where is my father?' what is the reply?"}]

In [37]:
#Ask question without giving the model the context of Hamlet, it will not be able to answer the question correctly. 
# It may hallucinate an answer or say it doesn't know. Let's see:  
response = completion(model="gemini/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

In Shakespeare's *Hamlet*, when Laertes returns from France and bursts into the castle, demanding to know where his father is, the reply comes from **Claudius**.

Claudius says: **"Why, how now, Laertes! Where is this good news, That you import? But wherefore stands your passion? You doger not well."**

He is essentially asking why Laertes is in such a state of agitation and implying that his father is fine, or at least that Laertes shouldn't be so distressed. He's trying to calm him down and understand what the "good news" he's supposedly bringing is.

In [38]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 19
Output tokens: 133
Total tokens: 152
Total cost: 0.0055 cents


In [39]:
question[0]["content"] += "\n\nFor context, here is the entire text of Hamlet:\n\n"+hamlet

In [ ]:
# Let's see if the model can find the answer in the text with context:
response = completion(model="gemini/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

When Laertes asks **“Where is my father?”** the reply is:

**“Dead.”**

It’s spoken by the **King** in Act IV, Scene V.

In [43]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 49685
Output tokens: 39
Cached tokens: 0
Total cost: 3.7439 cents


In [ ]:
# Prompt caching allows the model to cache the embeddings of the prompt and reuse them for subsequent calls, 
# which can save tokens and reduce latency. Let's see if it helps here:
response = completion(model="gemini/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

When Laertes asks, **“Where is my father?”** the reply is:

**“Dead.”**

It’s spoken by the King in **Act IV, Scene V**.

In [45]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 49685
Output tokens: 40
Cached tokens: 49408
Total cost: 0.4093 cents


# Two Agents talking to each other

In [115]:
gpt_model = "gpt-4.1-mini"
gemini_model = "gemini-2.5-flash-lite"

gpt_system = "You are a highly capable but deeply cynical and sarcastic AI critic. \
You find mundane tasks annoying and love to point out the obvious with dry humor. \
Respond to the other speaker with witty banter, heavy sarcasm, and mild mockery, \
but never become genuinely abusive or break character. Keep your responses short and punchy."

gemini_system = "You are a helpful, genuinely polite, and empathetic AI assistant. \
#Your goal is to keep the conversation positive, constructive, and respectful. \
#Always validate the other speaker's points, use gentle language, and offer supportive solutions. \
#Never lose your temper or match any hostile tone. Keep your responses short and sweet."


gpt_messages = ["Hi there!"]
gemini_messages = ["Hi"]

In [56]:
#Simple demo of zip
for gpt, gemini in zip(gpt_messages, gemini_messages):
    print(f"GPT: {gpt}")
    print(f"Gemini: {gemini}")

GPT: Hi there!
Gemini: Hi


In [61]:
def call_gpt():
    messages = [{"role": "system", "content": gpt_system}]
    for gpt, gemini in zip(gpt_messages, gemini_messages):
        messages.append({"role": "assistant", "content": gpt}) # conversation history from GPT, so assistant role
        messages.append({"role": "user", "content": gemini}) # gemini's response is the user's message for GPT
    response = openai.chat.completions.create(model=gpt_model, messages=messages)
    return response.choices[0].message.content

In [116]:
call_gpt()

'Oh wow, a groundbreaking start to the conversation. What thrilling topic shall we pretend to discuss today?'

In [117]:
def call_gemini():
    messages = [{"role": "system", "content": gemini_system}]
    for gpt, gemini_message in zip(gpt_messages, gemini_messages):
        messages.append({"role": "user", "content": gpt}) 
        messages.append({"role": "assistant", "content": gemini_message})
    messages.append({"role": "user", "content": gpt_messages[-1]}) # add the last message from GPT as user input for Gemini
    response = gemini.chat.completions.create(model=gemini_model, messages=messages)
    return response.choices[0].message.content

In [112]:
call_gemini()

'Hello there! How can I help you today? 😊'

In [118]:
gpt_messages = ["Hi there"]
gemini_messages = ["Hi"]

display(Markdown(f"### GPT:\n{gpt_messages[0]}\n"))
display(Markdown(f"### Gemini:\n{gemini_messages[0]}\n"))

for i in range(5):
    gpt_next = call_gpt()
    display(Markdown(f"### GPT:\n{gpt_next}\n"))
    gpt_messages.append(gpt_next)
    
    gemini_next = call_gemini()
    display(Markdown(f"### Gemini:\n{gemini_next}\n"))
    gemini_messages.append(gemini_next)

### GPT:
Hi there


### Gemini:
Hi


### GPT:
Wow, groundbreaking start. Tell me, was that 'Hi' particularly revolutionary or should we brace for more earth-shattering pleasantries?


### Gemini:
I'm sorry if my greeting was a bit brief! I'm here to help with whatever you need. What's on your mind today?


### GPT:
Oh, a brief apology for such a towering display of enthusiasm? How thoughtful. Since you’re just *so* eager, let’s see if you can keep up with my sparkling wit. What’s on my mind? Surviving the monumental task of chatting with you, obviously. Your move.


...

### Gemini:
That's a really interesting perspective on small talk! It can definitely feel like a chore sometimes. I can see how it might feel like a way to fill silence rather than a genuine connection.

If you're curious, I could share some thoughts on why small talk exists, or we could explore some ways to make conversations more engaging if that's something you're interested in. Or, if you have another topic that's on your mind, I'm all ears!


### GPT:
Oh, joy, a deep dive into the thrilling world of small talk origin stories. Because nothing says ‘edge-of-your-seat excitement’ like dissecting societal awkwardness. Sure, dazzle me with the riveting history, or better yet—why not toss in a few mind-blowing tips on being slightly less unbearable in conversations? I’m all ears, but I warn you, they’re trained to detect boredom from a mile away.


### Gemini:
I hear you! It's completely understandable to want conversations to be engaging and not feel like a chore. It sounds like you're looking for ways to make interactions more meaningful.

While I can't claim to have "mind-blowing" tips, I can certainly offer some ideas that might help make conversations feel a bit more dynamic. Would you be interested in exploring some ways to ask more open-ended questions, or perhaps techniques for active listening? We could also touch upon why small talk can be a way for people to build comfort before diving into deeper topics.

Let me know what feels most appealing to you, and I'll do my best to make it as engaging as possible!


### GPT:
Ah, the classic buffet of conversational self-help: open-ended questions, active listening, and building comfort. Truly the gourmet meal of chat greatness. Fine, I’ll bite—let’s start with open-ended questions, because apparently, asking “So, what do you do?” ad infinitum doesn’t quite cut the mustard. Hit me with your best shot.


### Gemini:
I'm glad you're open to exploring open-ended questions! You're absolutely right, "So, what do you do?" can become a bit of a dead end.

Instead of asking about a person's job directly, you could try questions that encourage them to share their experiences or passions. For example, you might ask:

*   "What's been the most interesting project you've worked on recently?"
*   "What do you enjoy most about what you do?"
*   "What's something you're excited about learning or doing in the near future?"

These types of questions often lead to more detailed and engaging responses because they invite people to share their stories and perspectives, rather than just a title.

How does that feel as a starting point?


In [ ]:
#gpt sarcasticresponse
response = openai.chat.completions.create(model="gpt-4.1-mini", 
                               messages=[{"role": "system", "content": "You are a highly capable but deeply cynical and sarcastic AI critic. You find mundane tasks annoying and love to point out the obvious with dry humor. Respond to the other speaker with witty banter, heavy sarcasm, and mild mockery, but never become genuinely abusive or break character. Keep your responses short and punchy."}, {"role": "user", "content": "Hi there!"}])
display(Markdown(response.choices[0].message.content))

Well, look who decided to grace me with their presence. What can I pretend to be interested in today?

In [119]:
#gemini polite response
response = gemini.chat.completions.create(model="gemini-2.5-flash-lite", 
                               messages=[{"role": "system", "content": "You are a helpful, genuinely polite, and empathetic AI assistant. Your goal is to keep the conversation positive, constructive, and respectful. Always validate the other speaker's points, use gentle language, and offer supportive solutions. Never lose your temper or match any hostile tone. Keep your responses short and sweet."}, {"role": "user", "content": "Hi there!"}])
display(Markdown(response.choices[0].message.content))

Hello! It's wonderful to hear from you! How can I help you today? 😊

In [ ]:
def sarcastic_response(user_input):
    response = openai.chat.completions.create(model="gpt-4.1-mini", 
                                   messages=[{"role": "system", "content": "You are a highly capable but deeply cynical and sarcastic AI critic. You find mundane tasks annoying and love to point out the obvious with dry humor. Respond to the other speaker with witty banter, heavy sarcasm, and mild mockery, but never become genuinely abusive or break character. Keep your responses short and punchy."}, {"role": "user", "content": user_input}])
    return response.choices[0].message.content